In [ ]:
# ============================================================
# NOTEBOOK 06
# 3D RESUNET EVALUATION
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve
)

In [ ]:
# ============================================================
# CHECK TENSORFLOW
# ============================================================

print("TensorFlow version:", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")

if gpus:
    print("GPU available:")
    for gpu in gpus:
        print(gpu)
else:
    print("No GPU detected.")

In [ ]:
# ============================================================
# PATH CONFIGURATION
# ============================================================

MODEL_ROOT = "/kaggle/input/3d-resunet-trained-model"

TEST_DATA_ROOT = "/kaggle/input/YOUR-TEST-DATASET"

MODEL_PATH = os.path.join(
    MODEL_ROOT,
    "best_3D_ResUNet.keras"
)

X_TEST_PATH = os.path.join(
    TEST_DATA_ROOT,
    "X_test.npy"
)

Y_SEG_TEST_PATH = os.path.join(
    TEST_DATA_ROOT,
    "Y_segmentation_test.npy"
)

Y_CLASS_TEST_PATH = os.path.join(
    TEST_DATA_ROOT,
    "Y_classification_test.npy"
)

print("Model:")
print(MODEL_PATH)

print()

print("Test input:")
print(X_TEST_PATH)

In [ ]:
# ============================================================
# VERIFY FILES
# ============================================================

required_files = [
    MODEL_PATH,
    X_TEST_PATH,
    Y_SEG_TEST_PATH,
    Y_CLASS_TEST_PATH
]

for path in required_files:

    if os.path.exists(path):

        print("FOUND:")
        print(path)

    else:

        print("NOT FOUND:")
        print(path)

In [ ]:
# ============================================================
# DICE COEFFICIENT
# ============================================================

def dice_coefficient(
    y_true,
    y_pred
):

    smooth = 1e-6

    y_true = tf.cast(
        y_true,
        tf.float32
    )

    y_pred = tf.cast(
        y_pred > 0.5,
        tf.float32
    )

    intersection = tf.reduce_sum(
        y_true * y_pred,
        axis=[1, 2, 3, 4]
    )

    denominator = (
        tf.reduce_sum(
            y_true,
            axis=[1, 2, 3, 4]
        )
        +
        tf.reduce_sum(
            y_pred,
            axis=[1, 2, 3, 4]
        )
    )

    dice = (
        2.0 * intersection + smooth
    ) / (
        denominator + smooth
    )

    return tf.reduce_mean(dice)

In [ ]:
# ============================================================
# IOU
# ============================================================

def iou_coefficient(
    y_true,
    y_pred
):

    smooth = 1e-6

    y_true = tf.cast(
        y_true,
        tf.float32
    )

    y_pred = tf.cast(
        y_pred > 0.5,
        tf.float32
    )

    intersection = tf.reduce_sum(
        y_true * y_pred,
        axis=[1, 2, 3, 4]
    )

    union = (
        tf.reduce_sum(
            y_true,
            axis=[1, 2, 3, 4]
        )
        +
        tf.reduce_sum(
            y_pred,
            axis=[1, 2, 3, 4]
        )
        -
        intersection
    )

    iou = (
        intersection + smooth
    ) / (
        union + smooth
    )

    return tf.reduce_mean(iou)

In [ ]:
# ============================================================
# LOAD BEST MODEL
# ============================================================

model = tf.keras.models.load_model(
    MODEL_PATH,
    custom_objects={
        "dice_coefficient":
            dice_coefficient,

        "iou_coefficient":
            iou_coefficient
    }
)

print("Model loaded successfully.")

print()

print("Model name:")
print(model.name)

In [ ]:
# ============================================================
# MODEL OUTPUTS
# ============================================================

print("Input shape:")
print(model.input_shape)

print()

print("Segmentation output:")
print(model.output_shape[0])

print()

print("Classification output:")
print(model.output_shape[1])

In [ ]:
# ============================================================
# LOAD TEST DATA
# ============================================================

X_test = np.load(
    X_TEST_PATH
)

Y_seg_test = np.load(
    Y_SEG_TEST_PATH
)

Y_class_test = np.load(
    Y_CLASS_TEST_PATH
)

print("X_test:")
print(X_test.shape)

print()

print("Y_seg_test:")
print(Y_seg_test.shape)

print()

print("Y_class_test:")
print(Y_class_test.shape)

In [ ]:
# ============================================================
# CONVERT DATA TYPES
# ============================================================

X_test = X_test.astype(
    np.float32
)

Y_seg_test = Y_seg_test.astype(
    np.float32
)

Y_class_test = Y_class_test.astype(
    np.float32
)

print(
    "Data converted to float32."
)

In [ ]:
# ============================================================
# VALIDATE TEST DATA
# ============================================================

assert X_test.ndim == 5

assert Y_seg_test.ndim == 5

assert Y_class_test.ndim == 1

assert X_test.shape[1:] == (
    64,
    64,
    64,
    1
)

assert Y_seg_test.shape[1:] == (
    64,
    64,
    64,
    1
)

assert X_test.shape[0] == (
    Y_seg_test.shape[0]
)

assert X_test.shape[0] == (
    Y_class_test.shape[0]
)

print(
    "Test data dimensions are valid."
)

In [ ]:
# ============================================================
# MODEL EVALUATION
# ============================================================

evaluation_results = model.evaluate(

    X_test,

    {
        "segmentation_output":
            Y_seg_test,

        "classification_output":
            Y_class_test
    },

    batch_size=1,

    verbose=1,

    return_dict=True
)

print("\nEvaluation results:")
print("--------------------------------")

for metric, value in evaluation_results.items():

    print(
        f"{metric}: {value:.6f}"
    )

In [ ]:
# ============================================================
# GENERATE PREDICTIONS
# ============================================================

seg_predictions, class_predictions = (
    model.predict(
        X_test,
        batch_size=1,
        verbose=1
    )
)

print(
    "Segmentation predictions:"
)

print(
    seg_predictions.shape
)

print()

print(
    "Classification predictions:"
)

print(
    class_predictions.shape
)

In [ ]:
# ============================================================
# BINARY SEGMENTATION PREDICTIONS
# ============================================================

seg_predictions_binary = (
    seg_predictions >= 0.5
).astype(
    np.uint8
)

print(
    "Binary segmentation predictions:"
)

print(
    seg_predictions_binary.shape
)

In [ ]:
# ============================================================
# BINARY CLASSIFICATION PREDICTIONS
# ============================================================

class_probabilities = (
    class_predictions.ravel()
)

class_predictions_binary = (
    class_probabilities >= 0.5
).astype(
    np.uint8
)

Y_class_true = (
    Y_class_test.ravel()
    .astype(np.uint8)
)

print(
    "Classification probabilities:"
)

print(
    class_probabilities[:10]
)

print()

print(
    "Classification predictions:"
)

print(
    class_predictions_binary[:10]
)

In [ ]:
# ============================================================
# SEGMENTATION DICE
# ============================================================

dice_scores = []

for i in range(
    len(Y_seg_test)
):

    true_mask = (
        Y_seg_test[i] > 0.5
    )

    pred_mask = (
        seg_predictions_binary[i] > 0
    )

    intersection = np.logical_and(
        true_mask,
        pred_mask
    ).sum()

    denominator = (
        true_mask.sum()
        +
        pred_mask.sum()
    )

    if denominator == 0:

        dice = 1.0

    else:

        dice = (
            2.0 * intersection
        ) / denominator

    dice_scores.append(
        dice
    )

dice_scores = np.array(
    dice_scores
)

print(
    "Mean Dice:",
    np.mean(dice_scores)
)

print(
    "Median Dice:",
    np.median(dice_scores)
)

print(
    "Standard deviation:",
    np.std(dice_scores)
)

In [ ]:
# ============================================================
# SEGMENTATION IOU
# ============================================================

iou_scores = []

for i in range(
    len(Y_seg_test)
):

    true_mask = (
        Y_seg_test[i] > 0.5
    )

    pred_mask = (
        seg_predictions_binary[i] > 0
    )

    intersection = np.logical_and(
        true_mask,
        pred_mask
    ).sum()

    union = np.logical_or(
        true_mask,
        pred_mask
    ).sum()

    if union == 0:

        iou = 1.0

    else:

        iou = (
            intersection / union
        )

    iou_scores.append(
        iou
    )

iou_scores = np.array(
    iou_scores
)

print(
    "Mean IoU:",
    np.mean(iou_scores)
)

print(
    "Median IoU:",
    np.median(iou_scores)
)

print(
    "Standard deviation:",
    np.std(iou_scores)
)

In [ ]:
# ============================================================
# SEGMENTATION PRECISION
# ============================================================

seg_true = (
    Y_seg_test.ravel() > 0.5
)

seg_pred = (
    seg_predictions_binary.ravel() > 0
)

seg_precision = precision_score(
    seg_true,
    seg_pred,
    zero_division=0
)

print(
    "Segmentation Precision:",
    seg_precision
)

In [ ]:
# ============================================================
# SEGMENTATION RECALL
# ============================================================

seg_recall = recall_score(
    seg_true,
    seg_pred,
    zero_division=0
)

print(
    "Segmentation Recall:",
    seg_recall
)

In [ ]:
# ============================================================
# SEGMENTATION F1 SCORE
# ============================================================

seg_f1 = f1_score(
    seg_true,
    seg_pred,
    zero_division=0
)

print(
    "Segmentation F1-score:",
    seg_f1
)

In [ ]:
# ============================================================
# CLASSIFICATION ACCURACY
# ============================================================

classification_accuracy = accuracy_score(
    Y_class_true,
    class_predictions_binary
)

print(
    "Classification Accuracy:",
    classification_accuracy
)

In [ ]:
# ============================================================
# CLASSIFICATION PRECISION
# ============================================================

classification_precision = precision_score(
    Y_class_true,
    class_predictions_binary,
    zero_division=0
)

print(
    "Classification Precision:",
    classification_precision
)

In [ ]:
# ============================================================
# CLASSIFICATION RECALL
# ============================================================

classification_recall = recall_score(
    Y_class_true,
    class_predictions_binary,
    zero_division=0
)

print(
    "Classification Recall:",
    classification_recall
)

In [ ]:
# ============================================================
# CLASSIFICATION F1 SCORE
# ============================================================

classification_f1 = f1_score(
    Y_class_true,
    class_predictions_binary,
    zero_division=0
)

print(
    "Classification F1-score:",
    classification_f1
)

In [ ]:
# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    Y_class_true,
    class_predictions_binary
)

print(
    "Confusion Matrix:"
)

print(cm)

In [ ]:
# ============================================================
# CONFUSION MATRIX COMPONENTS
# ============================================================

TN, FP, FN, TP = cm.ravel()

print("True Negative :", TN)
print("False Positive:", FP)
print("False Negative:", FN)
print("True Positive :", TP)

In [ ]:
# ============================================================
# SPECIFICITY
# ============================================================

specificity = (
    TN / (TN + FP)
    if (TN + FP) > 0
    else 0
)

print(
    "Specificity:",
    specificity
)

In [ ]:
# ============================================================
# SENSITIVITY
# ============================================================

sensitivity = (
    TP / (TP + FN)
    if (TP + FN) > 0
    else 0
)

print(
    "Sensitivity:",
    sensitivity
)

In [ ]:
# ============================================================
# CLASSIFICATION AUC
# ============================================================

if len(
    np.unique(Y_class_true)
) == 2:

    classification_auc = roc_auc_score(
        Y_class_true,
        class_probabilities
    )

else:

    classification_auc = np.nan

print(
    "Classification AUC:",
    classification_auc
)

In [ ]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

report = classification_report(

    Y_class_true,

    class_predictions_binary,

    target_names=[
        "No Fracture",
        "Fracture"
    ],

    zero_division=0
)

print(report)

In [ ]:
# ============================================================
# PLOT CONFUSION MATRIX
# ============================================================

plt.figure(
    figsize=(6, 5)
)

plt.imshow(
    cm
)

plt.title(
    "Classification Confusion Matrix"
)

plt.xlabel(
    "Predicted Class"
)

plt.ylabel(
    "True Class"
)

plt.xticks(
    [0, 1],
    ["No Fracture", "Fracture"]
)

plt.yticks(
    [0, 1],
    ["No Fracture", "Fracture"]
)

for i in range(2):

    for j in range(2):

        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# ROC CURVE
# ============================================================

if len(
    np.unique(Y_class_true)
) == 2:

    fpr, tpr, thresholds = roc_curve(
        Y_class_true,
        class_probabilities
    )

    auc_value = roc_auc_score(
        Y_class_true,
        class_probabilities
    )

    plt.figure(
        figsize=(8, 6)
    )

    plt.plot(
        fpr,
        tpr,
        label=f"AUC = {auc_value:.4f}"
    )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--"
    )

    plt.xlabel(
        "False Positive Rate"
    )

    plt.ylabel(
        "True Positive Rate"
    )

    plt.title(
        "ROC Curve for Rib Fracture Classification"
    )

    plt.legend()

    plt.grid(True)

    plt.show()

In [ ]:
# ============================================================
# SEGMENTATION RESULTS TABLE
# ============================================================

segmentation_results = pd.DataFrame({

    "Patch_ID":
        np.arange(
            len(dice_scores)
        ),

    "Dice":
        dice_scores,

    "IoU":
        iou_scores
})

segmentation_results.head()

In [ ]:
# ============================================================
# CLASSIFICATION RESULTS TABLE
# ============================================================

classification_results = pd.DataFrame({

    "Patch_ID":
        np.arange(
            len(Y_class_true)
        ),

    "True_Class":
        Y_class_true,

    "Predicted_Probability":
        class_probabilities,

    "Predicted_Class":
        class_predictions_binary
})

classification_results.head()

In [ ]:
# ============================================================
# MAIN RESULTS
# ============================================================

final_results = pd.DataFrame({

    "Metric": [

        "Segmentation Dice",
        "Segmentation IoU",
        "Segmentation Precision",
        "Segmentation Recall",
        "Segmentation F1",

        "Classification Accuracy",
        "Classification Precision",
        "Classification Sensitivity",
        "Classification Specificity",
        "Classification F1",
        "Classification AUC"
    ],

    "Value": [

        np.mean(dice_scores),
        np.mean(iou_scores),
        seg_precision,
        seg_recall,
        seg_f1,

        classification_accuracy,
        classification_precision,
        sensitivity,
        specificity,
        classification_f1,
        classification_auc
    ]
})

final_results

In [ ]:
# ============================================================
# SAVE EVALUATION RESULTS
# ============================================================

OUTPUT_DIR = "/kaggle/working/evaluation_results"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

final_results_path = os.path.join(
    OUTPUT_DIR,
    "final_evaluation_results.csv"
)

final_results.to_csv(
    final_results_path,
    index=False
)

print(
    "Results saved:"
)

print(
    final_results_path
)

In [ ]:
# ============================================================
# SAVE SEGMENTATION RESULTS
# ============================================================

segmentation_path = os.path.join(
    OUTPUT_DIR,
    "segmentation_results.csv"
)

segmentation_results.to_csv(
    segmentation_path,
    index=False
)

print(
    segmentation_path
)

In [ ]:
# ============================================================
# SAVE CLASSIFICATION RESULTS
# ============================================================

classification_path = os.path.join(
    OUTPUT_DIR,
    "classification_results.csv"
)

classification_results.to_csv(
    classification_path,
    index=False
)

print(
    classification_path
)

In [ ]:
# ============================================================
# SAVE MODEL PREDICTIONS
# ============================================================

np.save(
    os.path.join(
        OUTPUT_DIR,
        "segmentation_predictions.npy"
    ),
    seg_predictions
)

np.save(
    os.path.join(
        OUTPUT_DIR,
        "segmentation_predictions_binary.npy"
    ),
    seg_predictions_binary
)

np.save(
    os.path.join(
        OUTPUT_DIR,
        "classification_probabilities.npy"
    ),
    class_probabilities
)

np.save(
    os.path.join(
        OUTPUT_DIR,
        "classification_predictions.npy"
    ),
    class_predictions_binary
)

print(
    "Predictions saved successfully."
)

In [ ]:
# ============================================================
# SAVE TEST DATA FOR NOTEBOOK 07
# ============================================================

np.save(
    os.path.join(
        OUTPUT_DIR,
        "X_test.npy"
    ),
    X_test
)

np.save(
    os.path.join(
        OUTPUT_DIR,
        "Y_seg_test.npy"
    ),
    Y_seg_test
)

np.save(
    os.path.join(
        OUTPUT_DIR,
        "Y_class_test.npy"
    ),
    Y_class_test
)

print(
    "Test data saved for visualization."
)

In [ ]:
# ============================================================
# FINAL EVALUATION SUMMARY
# ============================================================

print("=" * 60)
print("NOTEBOOK 06 - FINAL EVALUATION")
print("=" * 60)

print()

print("SEGMENTATION PERFORMANCE")
print("-" * 40)

print(
    f"Dice       : {np.mean(dice_scores):.4f}"
)

print(
    f"IoU        : {np.mean(iou_scores):.4f}"
)

print(
    f"Precision  : {seg_precision:.4f}"
)

print(
    f"Recall     : {seg_recall:.4f}"
)

print(
    f"F1-score   : {seg_f1:.4f}"
)

print()

print("CLASSIFICATION PERFORMANCE")
print("-" * 40)

print(
    f"Accuracy   : {classification_accuracy:.4f}"
)

print(
    f"Precision  : {classification_precision:.4f}"
)

print(
    f"Sensitivity: {sensitivity:.4f}"
)

print(
    f"Specificity: {specificity:.4f}"
)

print(
    f"F1-scoren: {classification_f1:.4f}"
)

print(
    f"AUC : {classification_auc:.4f}"
)

print()

print("=" * 60)
print("EVALUATION COMPLETED")
print("=" * 60)